In [ ]:
!pip install -q librosa audiomentations soundfile
!pip install -q timm transformers torchmetrics torchsummary
!pip install -q scikit-learn imbalanced-learn seaborn plotly shap
!pip install -q grad-cam kaggle gdown

In [ ]:
import os, glob, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from pathlib import Path
import soundfile as sf
import librosa
import librosa.display
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
import timm
from transformers import ASTFeatureExtractor, ASTForAudioClassification

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, f1_score, precision_score,
                              recall_score, balanced_accuracy_score,
                              cohen_kappa_score, ConfusionMatrixDisplay,
                              average_precision_score)
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

import audiomentations as AA
import shap

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
os.makedirs('/content/icbhi', exist_ok=True)

# Option A: Kaggle API (recommended — clean structured version)
try:
    from google.colab import files
    print("Upload your kaggle.json:")
    files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d vbookshelf/respiratory-sound-database \
        -p /content/icbhi --unzip
    print("ICBHI 2017 downloaded via Kaggle")
except:
    # Option B: Direct download from official mirror
    print("Trying direct download...")
    !wget -q "https://bhichallenge.med.auth.gr/sites/default/files/ICBHI_final_database.zip" \
         -O /content/icbhi/ICBHI.zip
    !unzip -q /content/icbhi/ICBHI.zip -d /content/icbhi/
    print("Downloaded from ICBHI challenge mirror")

# Locate audio and annotation files
audio_files = glob.glob('/content/icbhi/**/*.wav', recursive=True)
txt_files   = glob.glob('/content/icbhi/**/*.txt', recursive=True)
print(f"\nAudio files found: {len(audio_files)}")
print(f"Annotation files: {len(txt_files)}")

# Find the diagnosis file (maps patient IDs to diagnoses)
diag_candidates = glob.glob('/content/icbhi/**/ICBHI_Challenge_diagnosis.txt', recursive=True) + \
                  glob.glob('/content/icbhi/**/*diagnosis*', recursive=True)
print(f"Diagnosis files: {diag_candidates}")


In [ ]:
def parse_icbhi_annotations(audio_files, txt_files, diag_file=None):
    """
    Parse ICBHI annotation format:
    Each .txt file has rows: start_time end_time crackle wheeze
    crackle/wheeze: 0=absent 1=present
    → 4 classes: Normal(0,0), Crackle(1,0), Wheeze(0,1), Both(1,1)
    """
    records = []

    # Build diagnosis map if available
    diag_map = {}
    if diag_file and os.path.exists(diag_file):
        with open(diag_file) as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    diag_map[parts[0]] = parts[1]

    # Map audio files to annotation files
    for audio_path in audio_files:
        base = Path(audio_path).stem
        ann_path = audio_path.replace('.wav', '.txt')
        if not os.path.exists(ann_path):
            # Try to find annotation in txt_files
            matching = [t for t in txt_files if Path(t).stem == base]
            if matching:
                ann_path = matching[0]
            else:
                continue

        # Parse patient info from filename: patientID_recordingIndex_chestLocation_acquisitionMode_equipmentType
        parts = base.split('_')
        patient_id = parts[0] if len(parts) > 0 else 'unknown'
        chest_loc  = parts[2] if len(parts) > 2 else 'unknown'
        acq_mode   = parts[3] if len(parts) > 3 else 'unknown'

        try:
            with open(ann_path) as f:
                for line in f:
                    row = line.strip().split('\t')
                    if len(row) < 4:
                        continue
                    start, end   = float(row[0]), float(row[1])
                    has_crackle  = int(row[2])
                    has_wheeze   = int(row[3])

                    # 4-class label
                    if has_crackle == 0 and has_wheeze == 0:
                        label = 0  # Normal
                    elif has_crackle == 1 and has_wheeze == 0:
                        label = 1  # Crackle
                    elif has_crackle == 0 and has_wheeze == 1:
                        label = 2  # Wheeze
                    else:
                        label = 3  # Both

                    records.append({
                        'audio_path':  audio_path,
                        'patient_id':  patient_id,
                        'start':       start,
                        'end':         end,
                        'crackle':     has_crackle,
                        'wheeze':      has_wheeze,
                        'label':       label,
                        'chest_loc':   chest_loc,
                        'acq_mode':    acq_mode,
                        'diagnosis':   diag_map.get(patient_id, 'Unknown'),
                        'duration':    end - start
                    })
        except Exception as e:
            continue

    return pd.DataFrame(records)

diag_file = diag_candidates[0] if diag_candidates else None
df = parse_icbhi_annotations(audio_files, txt_files, diag_file)

CLASS_NAMES  = {0: 'Normal', 1: 'Crackle', 2: 'Wheeze', 3: 'Both'}
CLASS_COLORS = {0: '#2ecc71', 1: '#e74c3c', 2: '#3498db', 3: '#9b59b6'}

print(f"Total respiratory cycles parsed: {len(df)}")
print(f"\nClass distribution:")
for k, v in CLASS_NAMES.items():
    count = (df['label'] == k).sum()
    pct   = count / len(df) * 100
    print(f"  {v:8s}: {count:4d} ({pct:.1f}%)")
print(f"\nUnique patients: {df['patient_id'].nunique()}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# Class distribution
counts = [( df['label'] == k).sum() for k in range(4)]
bars = axes[0,0].bar([CLASS_NAMES[k] for k in range(4)], counts,
                     color=[CLASS_COLORS[k] for k in range(4)],
                     edgecolor='black', alpha=0.85)
axes[0,0].set_title('ICBHI Cycle Class Distribution', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('Number of Cycles')
for bar, val in zip(bars, counts):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                   str(val), ha='center', fontweight='bold')
imbalance = max(counts) / max(min(counts), 1)
axes[0,0].set_xlabel(f'Imbalance ratio: {imbalance:.1f}x', fontsize=9)
axes[0,0].grid(True, alpha=0.3)

# Duration distribution
for k in range(4):
    durs = df[df['label'] == k]['duration']
    axes[0,1].hist(durs, bins=30, alpha=0.55, label=CLASS_NAMES[k],
                   color=CLASS_COLORS[k], density=True)
axes[0,1].set_title('Cycle Duration by Class', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Duration (seconds)')
axes[0,1].set_ylabel('Density')
axes[0,1].legend(fontsize=9)
axes[0,1].grid(True, alpha=0.3)

# Chest location distribution
if 'chest_loc' in df.columns:
    loc_counts = df.groupby(['chest_loc', 'label']).size().unstack(fill_value=0)
    loc_counts.plot(kind='bar', ax=axes[0,2],
                    color=[CLASS_COLORS[k] for k in range(4)],
                    alpha=0.85, edgecolor='black')
    axes[0,2].set_title('Cycles per Chest Location', fontsize=12, fontweight='bold')
    axes[0,2].set_xlabel('Location')
    axes[0,2].set_ylabel('Count')
    axes[0,2].tick_params(axis='x', rotation=30)
    axes[0,2].legend([CLASS_NAMES[k] for k in range(4)], fontsize=8)
    axes[0,2].grid(True, alpha=0.3)

# Diagnosis distribution
if 'diagnosis' in df.columns:
    diag_counts = df.groupby('diagnosis')['label'].value_counts().unstack(fill_value=0)
    diag_counts.head(8).plot(kind='barh', ax=axes[1,0],
                              color=[CLASS_COLORS[k] for k in range(4)], alpha=0.85)
    axes[1,0].set_title('Sound Classes per Diagnosis', fontsize=12, fontweight='bold')
    axes[1,0].set_xlabel('Count')
    axes[1,0].legend([CLASS_NAMES[k] for k in range(4)], fontsize=8)
    axes[1,0].grid(True, alpha=0.3)

# Cycles per patient
cycles_per_patient = df.groupby('patient_id').size()
axes[1,1].hist(cycles_per_patient, bins=25, color='#3498db',
               edgecolor='black', alpha=0.85)
axes[1,1].set_title('Cycles per Patient', fontsize=12, fontweight='bold')
axes[1,1].set_xlabel('Number of cycles')
axes[1,1].set_ylabel('Patients')
axes[1,1].axvline(cycles_per_patient.mean(), color='red', linestyle='--',
                   linewidth=2, label=f'Mean: {cycles_per_patient.mean():.1f}')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# Acquisition mode
if 'acq_mode' in df.columns:
    mode_counts = df['acq_mode'].value_counts()
    axes[1,2].pie(mode_counts.values, labels=mode_counts.index,
                  autopct='%1.1f%%', startangle=90,
                  colors=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'])
    axes[1,2].set_title('Acquisition Mode Distribution', fontsize=12, fontweight='bold')

plt.suptitle('ICBHI 2017: Exploratory Data Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_icbhi_overview.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
SR_TARGET = 16000  # standardize to 16kHz

def load_cycle(row, sr=SR_TARGET):
    """Load a single respiratory cycle from disk."""
    try:
        y, sr_orig = librosa.load(row['audio_path'], sr=sr, mono=True)
        start_s = int(row['start'] * sr)
        end_s   = int(row['end']   * sr)
        cycle   = y[start_s:end_s]
        if len(cycle) < sr * 0.5:   # skip cycles < 0.5 sec
            return None
        return cycle
    except:
        return None

# Sample one cycle per class
fig, axes = plt.subplots(4, 4, figsize=(22, 16))

for row_idx, (cls_id, cls_name) in enumerate(CLASS_NAMES.items()):
    cls_rows = df[df['label'] == cls_id]
    if cls_rows.empty:
        continue
    sample_row = cls_rows.iloc[0]
    cycle = load_cycle(sample_row)
    if cycle is None:
        continue

    t = np.arange(len(cycle)) / SR_TARGET
    color = CLASS_COLORS[cls_id]

    # Waveform
    axes[row_idx, 0].plot(t, cycle, color=color, linewidth=0.7)
    axes[row_idx, 0].set_title(f'{cls_name} — Waveform', fontweight='bold', color=color)
    axes[row_idx, 0].set_xlabel('Time (s)')
    axes[row_idx, 0].set_ylabel('Amplitude')
    axes[row_idx, 0].grid(True, alpha=0.3)

    # Mel Spectrogram
    mel = librosa.feature.melspectrogram(y=cycle, sr=SR_TARGET, n_mels=128,
                                          fmax=8000, n_fft=1024, hop_length=256)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img1 = librosa.display.specshow(mel_db, sr=SR_TARGET, x_axis='time',
                                     y_axis='mel', ax=axes[row_idx, 1], fmax=8000)
    axes[row_idx, 1].set_title(f'{cls_name} — Mel Spectrogram', fontweight='bold')
    plt.colorbar(img1, ax=axes[row_idx, 1], format='%+2.0f dB')

    # MFCC
    mfcc = librosa.feature.mfcc(y=cycle, sr=SR_TARGET, n_mfcc=20)
    img2 = librosa.display.specshow(mfcc, sr=SR_TARGET, x_axis='time',
                                     ax=axes[row_idx, 2])
    axes[row_idx, 2].set_title(f'{cls_name} — MFCC (20)', fontweight='bold')
    plt.colorbar(img2, ax=axes[row_idx, 2])

    # CQT (Constant-Q Transform) — captures harmonics better for wheeze
    cqt = np.abs(librosa.cqt(cycle, sr=SR_TARGET, hop_length=256))
    cqt_db = librosa.amplitude_to_db(cqt, ref=np.max)
    img3 = librosa.display.specshow(cqt_db, sr=SR_TARGET, x_axis='time',
                                     y_axis='cqt_hz', ax=axes[row_idx, 3])
    axes[row_idx, 3].set_title(f'{cls_name} — CQT', fontweight='bold')
    plt.colorbar(img3, ax=axes[row_idx, 3])

plt.suptitle('Respiratory Cycle Representations by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_waveforms_spectrograms.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def extract_audio_features(cycle, sr=SR_TARGET):
    """Extract handcrafted features for classical ML baseline."""
    features = {}

    # Zero crossing rate
    features['zcr_mean'] = librosa.feature.zero_crossing_rate(cycle).mean()
    features['zcr_std']  = librosa.feature.zero_crossing_rate(cycle).std()

    # RMS energy
    rms = librosa.feature.rms(y=cycle)
    features['rms_mean'] = rms.mean()
    features['rms_std']  = rms.std()

    # Spectral features
    spec_cent = librosa.feature.spectral_centroid(y=cycle, sr=sr)
    spec_bw   = librosa.feature.spectral_bandwidth(y=cycle, sr=sr)
    spec_roll = librosa.feature.spectral_rolloff(y=cycle, sr=sr)
    spec_flat = librosa.feature.spectral_flatness(y=cycle)
    features['spec_centroid_mean'] = spec_cent.mean()
    features['spec_centroid_std']  = spec_cent.std()
    features['spec_bandwidth_mean']= spec_bw.mean()
    features['spec_rolloff_mean']  = spec_roll.mean()
    features['spec_flatness_mean'] = spec_flat.mean()

    # MFCC (20 coefficients × mean + std = 40 features)
    mfcc = librosa.feature.mfcc(y=cycle, sr=sr, n_mfcc=20)
    for i, coef in enumerate(mfcc):
        features[f'mfcc_{i}_mean'] = coef.mean()
        features[f'mfcc_{i}_std']  = coef.std()

    # Chroma
    chroma = librosa.feature.chroma_stft(y=cycle, sr=sr)
    features['chroma_mean'] = chroma.mean()
    features['chroma_std']  = chroma.std()

    # Mel band energies (8 bands)
    mel = librosa.feature.melspectrogram(y=cycle, sr=sr, n_mels=8)
    for i in range(8):
        features[f'mel_band_{i}'] = mel[i].mean()

    # Statistical moments of waveform
    features['skewness'] = float(pd.Series(cycle).skew())
    features['kurtosis'] = float(pd.Series(cycle).kurtosis())

    return features

# Extract for 500 samples (fast preview)
print("Extracting audio features for EDA...")
sample_df = df.groupby('label').head(125).copy().reset_index(drop=True)
feat_list = []
for _, row in sample_df.iterrows():
    cycle = load_cycle(row)
    if cycle is not None:
        feats = extract_audio_features(cycle)
        feats['label'] = row['label']
        feat_list.append(feats)

feat_df = pd.DataFrame(feat_list)
print(f"Feature matrix: {feat_df.shape}")

# Plot key feature distributions
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
key_features = ['zcr_mean', 'rms_mean', 'spec_centroid_mean', 'spec_flatness_mean',
                'mfcc_0_mean', 'mfcc_1_mean', 'chroma_mean', 'spec_rolloff_mean']

for ax, feat in zip(axes.flatten(), key_features):
    for cls_id, cls_name in CLASS_NAMES.items():
        vals = feat_df[feat_df['label'] == cls_id][feat].dropna()
        if len(vals) > 0:
            ax.hist(vals, bins=20, alpha=0.55, label=cls_name,
                    color=CLASS_COLORS[cls_id], density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold', fontsize=10)
    ax.set_xlabel(feat, fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature Distributions by Respiratory Sound Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_features_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
CYCLE_DURATION = 5.0   # seconds — fixed-length window
CYCLE_SAMPLES  = int(CYCLE_DURATION * SR_TARGET)   # 80,000 samples

def pad_or_trim(signal, target_len):
    """Pad with zeros or trim to exactly target_len."""
    if len(signal) >= target_len:
        return signal[:target_len]
    pad = target_len - len(signal)
    return np.pad(signal, (0, pad), mode='constant')

def apply_bandpass(signal, sr, low=100, high=2000):
    """Bandpass filter — lung sounds are 100–2000 Hz."""
    from scipy import signal as sp_sig
    nyq = sr / 2
    b, a = sp_sig.butter(4, [low/nyq, high/nyq], btype='band')
    return sp_sig.filtfilt(b, a, signal)

def normalize_audio(signal):
    """Peak normalization."""
    peak = np.abs(signal).max()
    return signal / (peak + 1e-8)

# Audiomentations augmentation pipeline (training only)
train_augment = AA.Compose([
    AA.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.4),
    AA.TimeStretch(min_rate=0.85, max_rate=1.15, p=0.3),
    AA.PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
    AA.Shift(min_shift=-0.2, max_shift=0.2, p=0.3),
    AA.RoomSimulator(p=0.2),   # simulate auscultation conditions
    AA.Gain(min_gain_db=-6, max_gain_db=6, p=0.3),
])

def compute_features(signal, sr=SR_TARGET, feature_type='mel'):
    """Compute spectrogram features for CNN input."""
    signal = apply_bandpass(signal, sr)
    signal = normalize_audio(signal)
    signal = pad_or_trim(signal, CYCLE_SAMPLES)

    if feature_type == 'mel':
        feat = librosa.feature.melspectrogram(
            y=signal, sr=sr, n_mels=128, n_fft=1024,
            hop_length=256, fmax=2000, power=2.0)
        feat = librosa.power_to_db(feat, ref=np.max)

    elif feature_type == 'mfcc':
        feat = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=40,
                                     n_fft=1024, hop_length=256)

    elif feature_type == 'multi':
        # Triple-feature fusion: Mel + MFCC + Chromagram stacked
        mel  = librosa.power_to_db(
            librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=64,
                                            n_fft=1024, hop_length=256, fmax=2000),
            ref=np.max)
        mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=32,
                                     n_fft=1024, hop_length=256)
        chro = librosa.feature.chroma_stft(y=signal, sr=sr,
                                            n_fft=1024, hop_length=256)
        # Normalize each to [0,1] and stack
        def norm(x): return (x - x.min()) / (x.max() - x.min() + 1e-8)
        min_t = min(mel.shape[1], mfcc.shape[1], chro.shape[1])
        feat = np.stack([
            cv2.resize(norm(mel),  (min_t, 64)),
            cv2.resize(norm(mfcc), (min_t, 64)),
            cv2.resize(norm(chro), (min_t, 64))
        ], axis=0)  # (3, 64, T)
        return feat

    # Resize to fixed shape: (1, 128, 313) for mel/mfcc
    feat = cv2.resize(feat, (313, 128))
    return feat[np.newaxis, :, :]  # (1, 128, 313)

print("Preprocessing pipeline defined:")
print(f"  Target duration:  {CYCLE_DURATION}s  ({CYCLE_SAMPLES} samples at {SR_TARGET}Hz)")
print(f"  Bandpass filter:  100–2000 Hz (lung sound range)")
print(f"  Feature options:  mel (1×128×313) | mfcc | multi (3×64×T)")
print(f"  Augmentations:    Noise, TimeStretch, PitchShift, Shift, Room, Gain")


In [ ]:
# CRITICAL: split by PATIENT, not by cycle — prevents data leakage
patients = df['patient_id'].unique()
np.random.shuffle(patients)

n_test  = max(1, int(len(patients) * 0.2))
n_val   = max(1, int(len(patients) * 0.1))

test_patients  = patients[:n_test]
val_patients   = patients[n_test:n_test + n_val]
train_patients = patients[n_test + n_val:]

train_df = df[df['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df   = df[df['patient_id'].isin(val_patients)].reset_index(drop=True)
test_df  = df[df['patient_id'].isin(test_patients)].reset_index(drop=True)

print(f"Patient-level split:")
print(f"  Train: {len(train_patients)} patients | {len(train_df)} cycles")
print(f"  Val:   {len(val_patients)} patients | {len(val_df)} cycles")
print(f"  Test:  {len(test_patients)} patients | {len(test_df)} cycles")

for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"\n{split_name} class distribution:")
    for k, v in CLASS_NAMES.items():
        c = (split_df['label'] == k).sum()
        print(f"  {v:8s}: {c:4d}")


In [ ]:
class LungSoundDataset(Dataset):
    def __init__(self, df, feature_type='multi', augment=False, cache=True):
        self.df           = df.reset_index(drop=True)
        self.feature_type = feature_type
        self.augment      = augment
        self.cache        = {}
        self.do_cache     = cache

        # Pre-cache features for val/test to speed up training
        if cache and not augment:
            print(f"  Caching {len(df)} samples...")
            for idx in range(len(df)):
                self._load_and_process(idx)

    def _load_and_process(self, idx):
        if idx in self.cache:
            return self.cache[idx]
        row   = self.df.iloc[idx]
        cycle = load_cycle(row)
        if cycle is None:
            cycle = np.zeros(CYCLE_SAMPLES)
        if self.augment and np.random.rand() < 0.6:
            try:
                cycle = train_augment(samples=cycle.astype(np.float32),
                                      sample_rate=SR_TARGET)
            except:
                pass
        feat  = compute_features(cycle, feature_type=self.feature_type)
        label = int(row['label'])
        result = (torch.FloatTensor(feat), torch.LongTensor([label])[0])
        if self.do_cache and not self.augment:
            self.cache[idx] = result
        return result

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self._load_and_process(idx)

# Compute class weights for weighted sampler
labels_train = train_df['label'].values
class_sample_counts = np.bincount(labels_train)
weight_per_class = 1.0 / (class_sample_counts + 1e-8)
sample_weights   = torch.FloatTensor([weight_per_class[l] for l in labels_train])
weighted_sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

class_weights_tensor = torch.FloatTensor(
    compute_class_weight('balanced', classes=np.unique(labels_train), y=labels_train)
).to(device)

print("Creating datasets...")
train_dataset = LungSoundDataset(train_df, feature_type='multi', augment=True,  cache=False)
val_dataset   = LungSoundDataset(val_df,   feature_type='multi', augment=False, cache=True)
test_dataset  = LungSoundDataset(test_df,  feature_type='multi', augment=False, cache=True)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=weighted_sampler, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

print(f"\nTrain batches: {len(train_loader)}")
x_sample, y_sample = next(iter(train_loader))
print(f"Input shape: {x_sample.shape}  |  Labels: {y_sample.shape}")


In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation block — channel attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.se(x).unsqueeze(-1).unsqueeze(-1)

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Dropout2d(dropout)
        )
    def forward(self, x): return self.block(x)

class MultiFeatureLungCNN(nn.Module):
    """
    3-channel input: [Mel, MFCC, Chroma] stacked as RGB
    Achieves 97.25% on ICBHI per 2025 ASLD-Net paper with triple features
    """
    def __init__(self, n_classes=4, dropout=0.3):
        super().__init__()
        # Shared encoder
        self.encoder = nn.Sequential(
            ConvBNReLU(3, 32, k=3),
            SEBlock(32),
            nn.MaxPool2d(2, 2),                    # →(32, 64, 156)

            ConvBNReLU(32, 64, k=3),
            ConvBNReLU(64, 64, k=3),
            SEBlock(64),
            nn.MaxPool2d(2, 2),                    # →(64, 32, 78)

            ConvBNReLU(64, 128, k=3),
            ConvBNReLU(128, 128, k=3),
            SEBlock(128),
            nn.MaxPool2d(2, 2),                    # →(128, 16, 39)

            ConvBNReLU(128, 256, k=3),
            SEBlock(256),
            nn.AdaptiveAvgPool2d((4, 4))           # →(256, 4, 4)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        return self.classifier(self.encoder(x))

model_cnn = MultiFeatureLungCNN(n_classes=4).to(device)
print(f"MultiFeature CNN | Params: {sum(p.numel() for p in model_cnn.parameters()):,}")


In [ ]:
class PatchEmbed2D(nn.Module):
    """2D patch embedding for spectrogram → transformer tokens."""
    def __init__(self, img_h=64, img_w=313, patch_h=8, patch_w=8,
                 in_ch=3, embed_dim=256):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim,
                               kernel_size=(patch_h, patch_w),
                               stride=(patch_h, patch_w))
        n_h = img_h  // patch_h
        n_w = img_w  // patch_w
        self.n_patches = n_h * n_w

    def forward(self, x):
        x = self.proj(x)                   # (B, D, n_h, n_w)
        x = x.flatten(2).transpose(1, 2)   # (B, N, D)
        return x

class MultiStageLungTransformer(nn.Module):
    """
    CNN front-end + Transformer back-end
    Based on 2025 Multi-Stage Hybrid CNN-Transformer (MSHCT) architecture
    """
    def __init__(self, n_classes=4, d_model=256, nhead=8,
                 n_transformer_layers=4, dropout=0.2):
        super().__init__()
        # Stage 1: Local CNN feature extraction
        self.local_cnn = nn.Sequential(
            ConvBNReLU(3,  32, k=3),
            nn.MaxPool2d(2),
            ConvBNReLU(32, 64, k=3),
            nn.MaxPool2d(2),           # → (64, 16, 78)
        )
        # Stage 2: Patch embedding
        self.patch_embed = PatchEmbed2D(
            img_h=16, img_w=78, patch_h=4, patch_w=4,
            in_ch=64, embed_dim=d_model)

        # Stage 3: Transformer encoder
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_embed = nn.Parameter(
            torch.randn(1, self.patch_embed.n_patches + 1, d_model) * 0.02)
        self.pos_drop  = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True,
            norm_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(enc_layer,
                                                  num_layers=n_transformer_layers)

        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        # x: (B, 3, 64, T)
        local = self.local_cnn(x)                      # (B, 64, 16, T//4)
        tokens = self.patch_embed(local)               # (B, N, D)
        B = x.size(0)
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)       # (B, N+1, D)
        tokens = self.pos_drop(tokens + self.pos_embed[:, :tokens.size(1)])
        out = self.transformer(tokens)                 # (B, N+1, D)
        return self.classifier(self.norm(out[:, 0]))   # CLS token

model_transformer = MultiStageLungTransformer(n_classes=4).to(device)
print(f"CNN-Transformer | Params: {sum(p.numel() for p in model_transformer.parameters()):,}")


In [ ]:
# AST pretrained on AudioSet (527 classes) — best ICBHI performance per 2023-2025 papers
print("Loading pretrained AST from HuggingFace (MIT/ast-finetuned-audioset)...")

try:
    from transformers import AutoFeatureExtractor, ASTForAudioClassification, ASTConfig

    # Load AST pretrained on AudioSet-527 → fine-tune for our 4 classes
    ast_feature_extractor = AutoFeatureExtractor.from_pretrained(
        "MIT/ast-finetuned-audioset-10-10-0.4593")

    ast_config = ASTConfig.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
    ast_config.num_labels = 4
    ast_config.id2label   = {i: CLASS_NAMES[i] for i in range(4)}
    ast_config.label2id   = {v: k for k, v in CLASS_NAMES.items()}

    ast_model_base = ASTForAudioClassification.from_pretrained(
        "MIT/ast-finetuned-audioset-10-10-0.4593",
        config=ast_config,
        ignore_mismatched_sizes=True   # replace 527-class head with 4-class
    ).to(device)

    AST_AVAILABLE = True
    print("AST loaded successfully")
    print(f"AST params: {sum(p.numel() for p in ast_model_base.parameters()):,}")

except Exception as e:
    print(f"AST load failed: {e}")
    print("Falling back to EfficientNet-B2 with mel input as AST substitute")
    AST_AVAILABLE = False

    class EfficientNetLung(nn.Module):
        def __init__(self, n_classes=4):
            super().__init__()
            self.backbone = timm.create_model('efficientnet_b2', pretrained=True,
                                              in_chans=3, num_classes=0, global_pool='avg')
            self.head = nn.Sequential(
                nn.Linear(self.backbone.num_features, 256),
                nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(256, n_classes))
        def forward(self, x): return self.head(self.backbone(x))

    ast_model_base = EfficientNetLung(n_classes=4).to(device)


In [ ]:
class LungSoundASTDataset(Dataset):
    """Dataset that returns raw audio for AST feature extractor."""
    def __init__(self, df, feature_extractor=None, augment=False):
        self.df = df.reset_index(drop=True)
        self.fe = feature_extractor
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        cycle = load_cycle(row)
        if cycle is None:
            cycle = np.zeros(SR_TARGET * 3, dtype=np.float32)
        cycle = pad_or_trim(cycle.astype(np.float32), SR_TARGET * 5)
        cycle = normalize_audio(cycle)

        if self.augment and np.random.rand() < 0.5:
            try:
                cycle = train_augment(samples=cycle, sample_rate=SR_TARGET)
            except: pass

        label = int(row['label'])

        if self.fe is not None:
            inputs = self.fe(cycle, sampling_rate=SR_TARGET,
                             return_tensors='pt', padding=True)
            return inputs['input_values'].squeeze(0), torch.LongTensor([label])[0]
        else:
            # For EfficientNet fallback: compute mel
            feat = compute_features(cycle, feature_type='multi')
            return torch.FloatTensor(feat), torch.LongTensor([label])[0]

if AST_AVAILABLE:
    fe = ast_feature_extractor
    ast_train_ds = LungSoundASTDataset(train_df, fe, augment=True)
    ast_val_ds   = LungSoundASTDataset(val_df,   fe, augment=False)
    ast_test_ds  = LungSoundASTDataset(test_df,  fe, augment=False)
else:
    ast_train_ds = LungSoundDataset(train_df, 'multi', augment=True, cache=False)
    ast_val_ds   = LungSoundDataset(val_df,   'multi', augment=False, cache=True)
    ast_test_ds  = LungSoundDataset(test_df,  'multi', augment=False, cache=True)

ast_train_loader = DataLoader(ast_train_ds, batch_size=16, sampler=weighted_sampler, num_workers=2)
ast_val_loader   = DataLoader(ast_val_ds,   batch_size=16, shuffle=False, num_workers=2)
ast_test_loader  = DataLoader(ast_test_ds,  batch_size=16, shuffle=False, num_workers=2)


In [ ]:
def train_lung_model(model, model_name, train_loader, val_loader,
                     epochs=60, lr=1e-3, patience=10, is_ast=False):
    if is_ast and AST_AVAILABLE:
        # AST: lower LR for backbone, higher for head
        backbone_p = [p for n, p in model.named_parameters() if 'classifier' not in n]
        head_p     = [p for n, p in model.named_parameters() if 'classifier' in n]
        optimizer  = torch.optim.AdamW([
            {'params': backbone_p, 'lr': lr * 0.1},
            {'params': head_p,     'lr': lr}
        ], weight_decay=1e-4)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=15, T_mult=2, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    history = {'train_loss': [], 'val_loss': [], 'val_f1': [],
               'val_acc': [], 'val_se': [], 'val_sp': []}
    best_score = 0
    patience_ctr = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            if is_ast and AST_AVAILABLE:
                out = model(input_values=X_b).logits
            else:
                out = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses, all_preds, all_labels = [], [], []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                if is_ast and AST_AVAILABLE:
                    out = model(input_values=X_b).logits
                else:
                    out = model(X_b)
                val_losses.append(criterion(out, y_b).item())
                all_preds.extend(out.argmax(1).cpu().numpy())
                all_labels.extend(y_b.cpu().numpy())

        all_preds  = np.array(all_preds)
        all_labels = np.array(all_labels)

        # ICBHI score = (Se + Sp) / 2 — the official evaluation metric
        # Se = sensitivity on all abnormal (1+2+3 vs 0)
        # Sp = specificity on normal vs abnormal
        normal_mask = (all_labels == 0)
        se = ((all_preds != 0) & ~normal_mask).sum() / (~normal_mask).sum() if (~normal_mask).sum() > 0 else 0
        sp = ((all_preds == 0) &  normal_mask).sum() / ( normal_mask).sum() if ( normal_mask).sum() > 0 else 0
        icbhi_score = (se + sp) / 2

        val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

        history['train_loss'].append(np.mean(train_losses))
        history['val_loss'].append(np.mean(val_losses))
        history['val_f1'].append(val_f1)
        history['val_se'].append(se)
        history['val_sp'].append(sp)
        history['val_acc'].append(icbhi_score)

        scheduler.step()

        if icbhi_score > best_score:
            best_score = icbhi_score
            torch.save(model.state_dict(), f'best_{model_name}.pt')
            patience_ctr = 0
        else:
            patience_ctr += 1

        if (epoch + 1) % 10 == 0:
            print(f"[{model_name}] Epoch {epoch+1:3d} | "
                  f"Loss: {np.mean(train_losses):.4f} | "
                  f"ICBHI Score: {icbhi_score:.4f} | "
                  f"Se: {se:.3f} | Sp: {sp:.3f} | F1: {val_f1:.4f}")

        if patience_ctr >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(torch.load(f'best_{model_name}.pt'))
    print(f"  Best ICBHI Score: {best_score:.4f}")
    return model, history, best_score


In [ ]:
print("="*65)
print("Training Model 1: Multi-Feature CNN (Mel+MFCC+Chroma)")
print("="*65)
model_cnn, history_cnn, score_cnn = train_lung_model(
    model_cnn, 'lung_cnn', train_loader, val_loader,
    epochs=60, lr=5e-4)

print("\n" + "="*65)
print("Training Model 2: CNN-Transformer (Multi-Stage Hybrid)")
print("="*65)
model_transformer, history_transformer, score_transformer = train_lung_model(
    model_transformer, 'lung_transformer', train_loader, val_loader,
    epochs=60, lr=3e-4)

print("\n" + "="*65)
print("Training Model 3: AST (Audio Spectrogram Transformer)")
print("="*65)
ast_model_base, history_ast, score_ast = train_lung_model(
    ast_model_base, 'lung_ast',
    ast_train_loader, ast_val_loader,
    epochs=40, lr=1e-4, patience=8, is_ast=AST_AVAILABLE)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
histories = {
    'MultiFeature CNN':  history_cnn,
    'CNN-Transformer':   history_transformer,
    'AST':               history_ast
}
hist_colors = {'MultiFeature CNN': '#e74c3c',
               'CNN-Transformer': '#3498db',
               'AST': '#2ecc71'}

metrics_plot = [
    ('val_loss',  'Validation Loss'),
    ('val_acc',   'ICBHI Score = (Se+Sp)/2  ↑'),
    ('val_f1',    'Macro F1  ↑')
]
for ax, (metric, title) in zip(axes, metrics_plot):
    for name, hist in histories.items():
        if metric in hist:
            ax.plot(hist[metric], label=name,
                    color=hist_colors[name], linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Training History — Lung Sound Analyser', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves_lung.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def evaluate_lung_model(model, loader, model_name, is_ast=False):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b = X_b.to(device)
            if is_ast and AST_AVAILABLE:
                out = model(input_values=X_b).logits
            else:
                out = model(X_b)
            probs = F.softmax(out, dim=1).cpu().numpy()
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_probs.extend(probs)
            all_labels.extend(y_b.numpy())

    preds  = np.array(all_preds)
    probs  = np.array(all_probs)
    labels = np.array(all_labels)

    # ICBHI official metric: Se = sensitivity (all abnormal), Sp = specificity
    normal_mask = (labels == 0)
    se = ((preds != 0) & ~normal_mask).sum() / (~normal_mask).sum() if (~normal_mask).sum() > 0 else 0
    sp = ((preds == 0) &  normal_mask).sum() / ( normal_mask).sum() if ( normal_mask).sum() > 0 else 0

    metrics = {
        'ICBHI Score':    (se + sp) / 2,
        'Sensitivity':     se,
        'Specificity':     sp,
        'Balanced Acc':   balanced_accuracy_score(labels, preds),
        'Macro F1':       f1_score(labels, preds, average='macro', zero_division=0),
        'Weighted F1':    f1_score(labels, preds, average='weighted', zero_division=0),
        'Cohen Kappa':    cohen_kappa_score(labels, preds),
        'Macro AUROC':    roc_auc_score(labels, probs, multi_class='ovr', average='macro'),
        'Normal F1':      f1_score(labels==0, preds==0, zero_division=0),
        'Crackle F1':     f1_score(labels==1, preds==1, zero_division=0),
        'Wheeze F1':      f1_score(labels==2, preds==2, zero_division=0),
        'Both F1':        f1_score(labels==3, preds==3, zero_division=0),
    }

    print(f"\n{'='*55}")
    print(f"Model: {model_name}")
    print(f"{'='*55}")
    for k, v in metrics.items():
        flag = ' ← ICBHI official' if k == 'ICBHI Score' else ''
        print(f"  {k:20s}: {v:.4f}{flag}")
    print(f"\n{classification_report(labels, preds, target_names=list(CLASS_NAMES.values()))}")

    return preds, probs, labels, metrics

results = {}
for model_obj, model_name, is_ast in [
    (model_cnn,         'MultiFeature CNN',  False),
    (model_transformer, 'CNN-Transformer',   False),
    (ast_model_base,    'AST',               AST_AVAILABLE),
]:
    p, pr, l, m = evaluate_lung_model(model_obj, test_loader if not is_ast else ast_test_loader,
                                       model_name, is_ast)
    results[model_name] = {'preds': p, 'probs': pr, 'labels': l, 'metrics': m}


In [ ]:
from sklearn.preprocessing import label_binarize

fig, axes = plt.subplots(2, 3, figsize=(21, 13))
cls_labels = list(CLASS_NAMES.values())

for col_idx, (model_name, res) in enumerate(results.items()):
    preds, probs, labels = res['preds'], res['probs'], res['labels']

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                ax=axes[0, col_idx],
                xticklabels=cls_labels,
                yticklabels=cls_labels,
                linewidths=0.5)
    score = res['metrics']['ICBHI Score']
    f1    = res['metrics']['Macro F1']
    axes[0, col_idx].set_title(f'{model_name}\nICBHI={score:.3f} | F1={f1:.3f}',
                                fontweight='bold')
    axes[0, col_idx].set_xlabel('Predicted')
    axes[0, col_idx].set_ylabel('Actual')

    # ROC curves per class
    y_bin = label_binarize(labels, classes=[0, 1, 2, 3])
    for cls_id, cls_name in CLASS_NAMES.items():
        from sklearn.metrics import roc_curve
        fpr, tpr, _ = roc_curve(y_bin[:, cls_id], probs[:, cls_id])
        auc = roc_auc_score(y_bin[:, cls_id], probs[:, cls_id])
        axes[1, col_idx].plot(fpr, tpr, linewidth=1.5,
                               color=CLASS_COLORS[cls_id],
                               label=f'{cls_name} (AUC={auc:.3f})')
    axes[1, col_idx].plot([0,1],[0,1],'k--', linewidth=0.8)
    axes[1, col_idx].set_title(f'{model_name} — ROC', fontweight='bold')
    axes[1, col_idx].set_xlabel('FPR'); axes[1, col_idx].set_ylabel('TPR')
    axes[1, col_idx].legend(fontsize=8)
    axes[1, col_idx].grid(True, alpha=0.3)

plt.suptitle('Lung Sound Analyser — Confusion Matrices & ROC Curves',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_lung_sound.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

!pip install -q grad-cam

best_model_name = max(results, key=lambda k: results[k]['metrics']['ICBHI Score'])
best_model_obj  = {'MultiFeature CNN': model_cnn,
                   'CNN-Transformer': model_transformer,
                   'AST': ast_model_base}[best_model_name]

if 'CNN' in best_model_name:
    target_layer = [best_model_obj.encoder[-2]]  # last conv before pool
    cam = GradCAM(model=best_model_obj, target_layers=target_layer)

    fig, axes = plt.subplots(4, 4, figsize=(20, 18))
    for cls_id, cls_name in CLASS_NAMES.items():
        cls_samples = test_df[test_df['label'] == cls_id].head(2)
        for col_idx, (_, row) in enumerate(cls_samples.iterrows()):
            cycle = load_cycle(row)
            if cycle is None: continue

            feat = compute_features(cycle, feature_type='multi')
            x_tensor = torch.FloatTensor(feat).unsqueeze(0).to(device)

            grayscale_cam = cam(input_tensor=x_tensor)[0]

            # Display the Mel channel as background
            mel_img = feat[0]
            mel_norm = (mel_img - mel_img.min()) / (mel_img.max() - mel_img.min() + 1e-8)
            mel_rgb  = cv2.cvtColor((mel_norm * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB) / 255.0
            cam_img  = show_cam_on_image(mel_rgb, grayscale_cam, use_rgb=True)

            row_idx = cls_id * 2 + col_idx if cls_id * 2 + col_idx < 8 else 0
            ax_orig = axes[cls_id, col_idx * 2]
            ax_cam  = axes[cls_id, col_idx * 2 + 1]

            ax_orig.imshow(mel_norm, cmap='viridis', aspect='auto')
            ax_orig.set_title(f'{cls_name}\nMel Spectrogram', fontsize=9, fontweight='bold',
                               color=CLASS_COLORS[cls_id])
            ax_orig.axis('off')

            ax_cam.imshow(cam_img, aspect='auto')
            ax_cam.set_title(f'{cls_name}\nGradCAM', fontsize=9, fontweight='bold')
            ax_cam.axis('off')

    plt.suptitle(f'GradCAM — {best_model_name}: Where the Model Listens',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('gradcam_lung_sound.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
best_model_name = max(results, key=lambda k: results[k]['metrics']['ICBHI Score'])
best_model_obj  = {'MultiFeature CNN': model_cnn,
                   'CNN-Transformer': model_transformer,
                   'AST': ast_model_base}[best_model_name]

print(f"Best model: {best_model_name}")
print(f"  ICBHI Score: {results[best_model_name]['metrics']['ICBHI Score']:.4f}")
print(f"  Sensitivity: {results[best_model_name]['metrics']['Sensitivity']:.4f}")
print(f"  Specificity: {results[best_model_name]['metrics']['Specificity']:.4f}")
print(f"  Macro F1:    {results[best_model_name]['metrics']['Macro F1']:.4f}")
print(f"  AUROC:       {results[best_model_name]['metrics']['Macro AUROC']:.4f}")

torch.save({
    'model_state_dict': best_model_obj.state_dict(),
    'model_name':       best_model_name,
    'task':             'lung_sound_classification',
    'n_classes':        4,
    'classes':          CLASS_NAMES,
    'sampling_rate':    SR_TARGET,
    'cycle_duration_s': CYCLE_DURATION,
    'bandpass_hz':      [100, 2000],
    'feature_type':     'multi',
    'metrics':          results[best_model_name]['metrics'],
    'is_ast':           AST_AVAILABLE and best_model_name == 'AST',
    'trained_at':       time.strftime('%Y-%m-%d %H:%M:%S'),
}, 'medverse_lung_sound_analyser.pt')

with open('medverse_lung_config.json', 'w') as f:
    json.dump({
        'model':          best_model_name,
        'sampling_rate':  SR_TARGET,
        'cycle_sec':      CYCLE_DURATION,
        'bandpass':       {'low_hz': 100, 'high_hz': 2000},
        'classes':        {str(k): v for k, v in CLASS_NAMES.items()},
        'icbhi_score':    results[best_model_name]['metrics']['ICBHI Score'],
        'severity_map': {
            'Normal':  'normal',
            'Crackle': 'watch',   # → COPD, pneumonia, fibrosis
            'Wheeze':  'watch',   # → asthma, bronchitis, COPD
            'Both':    'critical' # → severe multi-pathology
        },
        'clinical_associations': {
            'Crackle': ['Pneumonia', 'Heart failure', 'Pulmonary fibrosis', 'COPD'],
            'Wheeze':  ['Asthma', 'COPD', 'Bronchitis', 'Foreign body'],
            'Both':    ['Severe COPD', 'Acute pulmonary oedema']
        },
        'hardware_mapping': 'I2S MEMS microphone (MAX98357A) on MedVerse vest'
    }, f, indent=2)

print("\nSaved:")
print("  medverse_lung_sound_analyser.pt   → models/lung/lung_analyser/weights.pt")
print("  medverse_lung_config.json")


In [ ]:
def predict_lung_sounds(audio_array, sr=SR_TARGET, is_ast_model=False):
    """
    Simulates MedVerse I2S microphone stream inference.
    audio_array: numpy float32 array from MAX98357A I2S mic
    sr: 16kHz (matches MedVerse vest audio pipeline)

    Returns structured JSON matching MedVerse /api/lung endpoint format.
    """
    # Segment into cycles
    cycle_len = int(CYCLE_DURATION * sr)
    n_cycles  = max(1, len(audio_array) // cycle_len)
    cycle_preds = []

    model_to_use = best_model_obj
    model_to_use.eval()

    for i in range(n_cycles):
        start = i * cycle_len
        cycle = audio_array[start:start + cycle_len]
        if len(cycle) < cycle_len // 2:
            continue

        cycle = pad_or_trim(cycle.astype(np.float32), cycle_len)
        cycle = apply_bandpass(cycle, sr)
        cycle = normalize_audio(cycle)

        if is_ast_model and AST_AVAILABLE:
            inputs = ast_feature_extractor(cycle, sampling_rate=sr,
                                            return_tensors='pt', padding=True)
            x = inputs['input_values'].to(device)
            with torch.no_grad():
                logits = model_to_use(input_values=x).logits
        else:
            feat = compute_features(cycle, feature_type='multi')
            x    = torch.FloatTensor(feat).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = model_to_use(x)

        probs     = F.softmax(logits, dim=1).cpu().numpy()[0]
        pred_cls  = probs.argmax()
        cycle_preds.append({
            'cycle':          i + 1,
            'prediction':     CLASS_NAMES[pred_cls],
            'confidence':     float(probs.max()),
            'probabilities': {CLASS_NAMES[k]: float(probs[k]) for k in range(4)},
        })

    # Aggregate across cycles — majority vote
    if cycle_preds:
        all_cls = [p['prediction'] for p in cycle_preds]
        from collections import Counter
        majority = Counter(all_cls).most_common(1)[0][0]
    else:
        majority = 'Normal'

    severity_map = {'Normal': 'normal', 'Crackle': 'watch',
                    'Wheeze': 'watch',  'Both': 'critical'}
    clinical_map = {
        'Normal':  'No adventitious sounds detected',
        'Crackle': 'Crackles suggest fluid/fibrosis: rule out pneumonia, HF, COPD',
        'Wheeze':  'Wheeze suggests obstruction: rule out asthma, COPD, bronchitis',
        'Both':    'Both crackles & wheeze: consider severe COPD or pulmonary oedema'
    }

    return {
        'overall_classification': majority,
        'severity':               severity_map[majority],
        'clinical_note':          clinical_map[majority],
        'cycle_results':          cycle_preds,
        'n_cycles_analysed':      len(cycle_preds),
        'model':                  best_model_name,
        'hardware':               'I2S MAX98357A MEMS mic @ 16kHz'
    }

# Test with a sample from test set
test_row   = test_df.iloc[0]
test_cycle = load_cycle(test_row)
if test_cycle is not None:
    result = predict_lung_sounds(test_cycle,
                                  is_ast_model=(AST_AVAILABLE and best_model_name=='AST'))
    print("=== MedVerse Inference — Lung Sound Analyser ===")
    print(f"  True label:  {CLASS_NAMES[test_row['label']]}")
    for k, v in result.items():
        if k != 'cycle_results':
            print(f"  {k}: {v}")
